In [2]:
import polars as pl
import datetime
from polars import DataFrame

from etl.aml.utils import *

In [3]:
dataset = '../../data/raw/HI-Medium_Trans.csv'

trans = (
    pl.read_csv(dataset)
    .with_columns(
        pl.col("Timestamp").str.strptime(pl.Datetime, '%Y/%m/%d %H:%M', strict=True)
    )
    .sort('Timestamp')
)

In [4]:
trans.head()

Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-09-01 00:00:00,1046,"""800A37D90""",274159,"""820C04F20""",26.42,"""US Dollar""",26.42,"""US Dollar""","""Credit Card""",0
2022-09-01 00:00:00,21418,"""800AB4DE0""",21418,"""800AB4DE0""",23.61,"""US Dollar""",23.61,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,32248,"""80056BBD0""",11,"""800BAFE20""",12373.74,"""US Dollar""",12373.74,"""US Dollar""","""ACH""",0
2022-09-01 00:00:00,11798,"""80145BAF0""",11798,"""80145BAF0""",4981.27,"""US Dollar""",4981.27,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,1924,"""800CCE420""",1924,"""800CCE420""",22.94,"""US Dollar""",22.94,"""US Dollar""","""Reinvestment""",0


In [5]:
zero = trans['Timestamp'].min()
hundred = trans['Timestamp'].max()

In [6]:
diff = hundred - zero
days = diff.days
sixty = zero + datetime.timedelta(days=days * 0.6)
eighty = zero + datetime.timedelta(days=days * 0.8)
hundred = zero + datetime.timedelta(days=days)

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-17 04:48:00 2022-09-22 14:24:00 2022-09-28 00:00:00


# Convert Currencies

In [7]:
converted = convert_currencies_to_usd(trans)

In [8]:
assert converted.shape[0] == trans.shape[0]
trans = converted

In [9]:
trans.head()

Timestamp,From Bank,From,To Bank,To,Payment Format,Is Laundering,Amount
datetime[μs],i64,str,i64,str,str,i64,f64
2022-09-01 00:00:00,1046,"""800A37D90""",274159,"""820C04F20""","""Credit Card""",0,26.42
2022-09-01 00:00:00,21418,"""800AB4DE0""",21418,"""800AB4DE0""","""Reinvestment""",0,23.61
2022-09-01 00:00:00,32248,"""80056BBD0""",11,"""800BAFE20""","""ACH""",0,12373.74
2022-09-01 00:00:00,11798,"""80145BAF0""",11798,"""80145BAF0""","""Reinvestment""",0,4981.27
2022-09-01 00:00:00,1924,"""800CCE420""",1924,"""800CCE420""","""Reinvestment""",0,22.94


# Format data to remove strings

In [10]:
ssl = remove_strings(trans, currencies=False)

In [11]:
assert ssl.shape[0] == trans.shape[0]

In [12]:
ssl.head()

Timestamp,From Bank,To Bank,Is Laundering,Amount,From,To,Payment Format
datetime[μs],i64,i64,i64,f64,u32,u32,u32
2022-09-01 00:00:00,1046,274159,0,26.42,416105,1785175,1
2022-09-01 00:00:00,21418,21418,0,23.61,846539,846539,5
2022-09-01 00:00:00,32248,11,0,12373.74,1163525,1388572,4
2022-09-01 00:00:00,11798,11798,0,4981.27,1799594,1799594,5
2022-09-01 00:00:00,1924,1924,0,22.94,1037786,1037786,5


# Global vars

In [13]:
def _prep(df: pl.DataFrame) -> pl.LazyFrame:
    return (
        df.lazy()
        .filter(
             pl.col('Timestamp').is_not_null()
             & pl.col('From').is_not_null()
             & pl.col('To').is_not_null()
        )
    )

# Temporal stats

In [14]:
def add_temporal_stats(df: pl.DataFrame) -> pl.DataFrame:
    df = _prep(df).select(
        pl.col('Timestamp'),
        pl.col('From'),
        pl.col('To'),
    )

    in_gaps = (
        df.sort(["Timestamp", "To"])
        .group_by(["To"])
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_in_sec"),
            pl.col("gap").list.var().alias("var_gap_in_sec"),
        ])
        .select(['To', "mu_gap_in_sec", "var_gap_in_sec"])
        .rename({'To': 'Node'})
    )

    out_gaps = (
        df.sort(["Timestamp", "From" ])
        .group_by("From")
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_out_sec"),
            pl.col("gap").list.var().alias("var_gap_out_sec"),
        ])
        .select(['From', "mu_gap_out_sec", "var_gap_out_sec"])
        .rename({'From': 'Node'})
    )

    return (
        in_gaps
        .join(out_gaps, on="Node", how="full", coalesce=True)
        .fill_null(-1)
        .collect()
    )

In [15]:
temporal_features = add_temporal_stats(ssl)

In [16]:
assert temporal_features.shape[0] == 2076999

In [17]:
temporal_features

Node,mu_gap_in_sec,var_gap_in_sec,mu_gap_out_sec,var_gap_out_sec
u32,f64,f64,f64,f64
340316,56840.0,5.4394e9,3717.567568,2.8375e7
1198638,0.0,null,396480.0,3.1439e11
807982,39120.0,null,null,null
301108,null,null,448920.0,1.1048e11
1161794,17130.0,5.34645e8,96154.285714,2.4622e10
…,…,…,…,…
562842,213460.0,1.3379e11,null,null
819299,360.0,null,null,null
1396000,27013.2,1.9590e9,null,null


# Fraudulent patterns

## 2-cycles

In [18]:
def count_reciprocal_neighbours(df: DataFrame) -> DataFrame:
    lf = _prep(df)

    edges = (
        lf.select(["From", "To"])
        .filter(pl.col("From") != pl.col("To"))
        .group_by(["From", "To"])
        .agg(pl.len().alias("w"))
    )

    recip = (
        edges.join(
            edges,
            left_on=["From", "To"],
            right_on=["To", "From"],
            how="inner",
            suffix="_rev",
        )
        .with_columns(
            pl.min_horizontal("w", "w_rev").alias("cycle_count")
        )
        .group_by("From")
        .agg(
            pl.col("cycle_count").sum().alias("r_2cycle")
        )
        .rename({"From": "Node"})
    )

    nodes = (
        pl.concat([
            lf.select(pl.col("From").alias("Node")),
            lf.select(pl.col("To").alias("Node")),
        ])
        .unique()
    )

    return (
        nodes.join(recip, on="Node", how="left")
        .with_columns(
            pl.col("r_2cycle").fill_null(0).cast(pl.Int64)
        )
        .collect()
    )

In [19]:
two_cycles = count_reciprocal_neighbours(ssl)

In [20]:
two_cycles.describe()

statistic,Node,r_2cycle
str,f64,f64
"""count""",2.076999e6,2.076999e6
"""null_count""",0.0,0.0
"""mean""",1.038499e6,0.033182
"""std""",599578.110216,0.239884
"""min""",0.0,0.0
"""25%""",519250.0,0.0
"""50%""",1.038499e6,0.0
"""75%""",1.557749e6,0.0
"""max""",2.076998e6,40.0


In [21]:
assert two_cycles.shape[0] == 2076999

In [22]:
two_cycles

Node,r_2cycle
u32,i64
454370,0
1870665,0
750208,0
107397,0
201366,0
…,…
1408408,0
433204,0
971938,0


## Ego profile

In [23]:
EPS = 1e-12

def compute_ego_profiles(trans: DataFrame) -> DataFrame:
    lf = _prep(trans)

    out_stats = (
        lf.group_by("From")
        .agg(
            pl.len().alias("deg_out"),
            pl.col("To").n_unique().alias("fan_out"),
            pl.col("Amount").sum().alias("vol_out"),
        )
        .rename({"From": "Node"})
    )

    in_stats = (
        lf.group_by("To")
        .agg(
            pl.len().alias("deg_in"),
            pl.col("From").n_unique().alias("fan_in"),
            pl.col("Amount").sum().alias("vol_in"),
        )
        .rename({"To": "Node"})
    )

    ego = (
        out_stats.join(in_stats, on="Node", how="full")
        .with_columns(
            pl.col("deg_out").fill_null(0).cast(pl.Int64),
            pl.col("fan_out").fill_null(0).cast(pl.Int64),
            pl.col("vol_out").fill_null(0.0),
            pl.col("deg_in").fill_null(0).cast(pl.Int64),
            pl.col("fan_in").fill_null(0).cast(pl.Int64),
            pl.col("vol_in").fill_null(0.0),
        )
        .with_columns(
            ((pl.col("vol_in") - pl.col("vol_out"))/ (pl.col("vol_in") + pl.col("vol_out") + pl.lit(EPS))).alias("flow_imbalance")
        )
        .with_columns(
            pl.when(pl.col('Node').is_null()).then(pl.col('Node_right')).otherwise(pl.col('Node')).alias('Node'),
        )
        .drop('Node_right')
    )

    return ego.collect()

In [24]:
ego_profile = compute_ego_profiles(ssl)

In [25]:
ego_profile

Node,deg_out,fan_out,vol_out,deg_in,fan_in,vol_in,flow_imbalance
u32,i64,i64,f64,i64,i64,f64,f64
1431030,1,1,8388.643477,1,1,8388.643477,0.0
464192,4,2,2096.944764,0,0,0.0,-1.0
1044599,1,1,2613.45,1,1,2613.45,0.0
2030724,1,1,1502.343837,33,6,191982.90068,0.984471
94547,61,3,12650.401375,73,3,247113.937691,0.902601
…,…,…,…,…,…,…,…
1266919,0,0,0.0,27,3,93030.234492,1.0
521323,0,0,0.0,85,3,6.6173e7,1.0
937006,0,0,0.0,3,1,639.87,1.0


In [26]:
assert ego_profile.shape[0] == 2076999

# Flow-aware positional prediction

In [53]:
EPS = 1e-12

def flow_targets_out_entropy_amount(trans: DataFrame, k2: bool = True) -> DataFrame:
    lf = _prep(trans).select(["From", "To", "Amount"]).lazy()

    W = (
        lf.group_by(["From", "To"])
        .agg(pl.col("Amount").sum().alias("w"))
    )

    P = (
        W.with_columns(
            pl.col("w").sum().over("From").alias("out_wsum")
        )
        .with_columns(
            (pl.col("w") / (pl.col("out_wsum") + pl.lit(EPS))).alias("p")
        )
        .select(["From", "To", "p"])
        .cache()
    )

    one = (
        P.group_by(["From"])
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_1_amt"),
            pl.len().alias("supp_out_1_amt"),
            pl.col("p").max().alias("pmax_out_1_amt"),
        )
        .rename({"From": "Node"})
    )

    if not k2:
        return one.collect()

    P1 = P.rename({"From": "src", "To": "mid", "p": "p1"})
    P2 = P.rename({"From": "mid", "To": "dst", "p": "p2"})

    two_step = (
        P1.join(P2, on="mid", how="inner")
        .with_columns((pl.col("p1") * pl.col("p2")).alias("p"))
        .group_by(["src", "dst"])
        .agg(pl.col("p").sum().alias("p"))
    )

    two = (
        two_step.group_by("src")
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_2_amt"),
            pl.len().alias("supp_out_2_amt"),
            pl.col("p").max().alias("pmax_out_2_amt"),
        )
        .rename({"src": "Node"})
    )

    return one.join(two, on="Node", how="left").collect()


In [ ]:
train_pos_pred = flow_targets_out_entropy_amount(ssl)

# Join

In [48]:
node_features = (
    temporal_features
    .join(train_pos_pred, on="Node", how="full", coalesce=True)
    .join(ego_profile, on="Node", how="full", coalesce=True)
    .join(two_cycles, on="Node", how="full", coalesce=True)
    .with_columns(
        pl.lit(1).alias("Feature"),
    )
)
node_features

window_start,Node,mu_gap_in_sec,var_gap_in_sec,mu_gap_out_sec,var_gap_out_sec,deg_out,fan_out,vol_out,deg_in,fan_in,vol_in,flow_imbalance,n_currencies_in,currency_entropy_in,top_currency_share_in,n_currencies_out,currency_entropy_out,top_currency_share_out,r_2cycle,Feature
i64,u32,f64,f64,f64,f64,i64,i64,f64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,i32
10,791995,20180.0,9.3695568e8,null,null,1,1,8.86,7,3,7104.64,0.997509,1,-9.9987e-13,1.0,1,-8.8707e-13,1.0,0,1
10,1159975,15630.0,3.418692e8,null,null,1,1,90.19,5,2,576.65,0.7295,1,-9.9831e-13,1.0,1,-9.8899e-13,1.0,0,1
10,482968,12997.5,5.6714805e8,57240.0,null,2,2,9397.98,9,2,56541.38,0.714951,1,-1.0001e-12,1.0,1,-9.9987e-13,1.0,0,1
10,297314,null,null,null,null,1,1,34867.68,1,1,34867.68,0.0,1,-1.0001e-12,1.0,1,-1.0001e-12,1.0,0,1
10,1132308,null,null,29010.0,1.6762e9,3,3,1.0883e6,1,1,623609.47,-0.271447,1,-1.0001e-12,1.0,1,-1.0001e-12,1.0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2246410,697895,null,null,0.0,null,2,2,104222.03,1,1,13539.55,-0.770051,1,-9.9987e-13,1.0,2,0.386217,0.870089,0,1
2246410,871229,13560.0,null,4520.0,6.12912e7,4,3,78485.52,2,1,67563.07,-0.074786,2,0.190095,0.95283,3,0.567086,0.82023,0,1
2246410,954955,32700.0,1.9947e9,12262.5,8.5632e8,9,5,166084.42,4,1,118652.97,-0.16658,2,0.556704,0.755063,3,0.995869,0.539427,0,1


In [49]:
trans = (
    ssl.with_columns(
        (pl.col("Timestamp") - pl.col("Timestamp").min())
        .dt.total_seconds()
        .cast(pl.Int64)
        .add(10)
        .alias("Timestamp"),
    )
    .sort("Timestamp")
    .with_row_index("Edge IDq")
)
trans

Edge IDq,Timestamp,From Bank,To Bank,Amount Received,Amount Paid,Is Laundering,From,To,Payment Format,Payment Currency,Receiving Currency
u32,i64,i64,i64,f64,f64,i64,u32,u32,u32,u32,u32
0,10,1046,274159,26.42,26.42,0,1635842,1600487,1,9,9
1,10,21418,21418,23.61,23.61,0,798353,798353,2,9,9
2,10,32248,11,12373.74,12373.74,0,1748257,799071,6,9,9
3,10,11798,11798,4981.27,4981.27,0,1630856,1630856,2,9,9
4,10,1924,1924,22.94,22.94,0,499983,499983,2,9,9
…,…,…,…,…,…,…,…,…,…,…,…
31898233,2375950,211767,219853,7339.28,7339.28,1,954955,1051471,6,6,6
31898234,2376850,211767,211767,5869.16,6877.38,0,954955,954955,6,9,2
31898235,2376850,211767,130596,5869.16,5869.16,1,954955,482179,6,2,2


# Write

In [50]:
node_features.write_csv('../data/HI-Medium_SSL_Nodes_whole_convert.csv')
trans.write_csv('../data/HI-Medium_SSL_Trans_whole_convert.csv')